# Few-Shot RAG with AWS

## 📖 Overview

This notebook demonstrates **Few-Shot RAG** using AWS services:
- **Example-guided generation** with sample Q&A pairs
- **Consistent formatting** across answers
- **Style and tone control** through examples
- **Domain-specific patterns** learned from examples

### What is Few-Shot RAG?

**Zero-Shot RAG (No Examples):**
```
Query → Retrieve → Generate
│
└─ LLM uses general knowledge only
```

**Few-Shot RAG (With Examples):**
```
Query → Retrieve → Add Example Q&A Pairs → Generate
                          ↓
                    Examples show:
                    - Desired format
                    - Answer style
                    - Level of detail
                    - Structure
```

### Key Concepts

1. **Few-Shot Learning**: Learn from examples
   - Provide 2-5 example Q&A pairs
   - LLM infers desired pattern
   - Applies pattern to new query
   - No fine-tuning needed

2. **Example Selection**: Choose good examples
   - Representative of desired style
   - Similar to target query (optional)
   - Clear and well-structured
   - Diverse but consistent

3. **Prompt Structure**:
   ```
   Examples:
   Q: Example question 1
   A: Example answer 1
   
   Q: Example question 2
   A: Example answer 2
   
   Now answer:
   Q: User's question
   A: [Generate following pattern]
   ```

4. **Use Cases**:
   - Consistent formatting (structured answers)
   - Specific tone/style (formal, casual, technical)
   - Domain conventions (medical, legal, technical)
   - Answer templates (always include X, Y, Z)

### Why Few-Shot?

**Problems it Solves:**
- ❌ Inconsistent answer formats
- ❌ Wrong tone or style
- ❌ Missing expected sections
- ❌ Need fine-tuning (expensive/slow)
- ❌ Hard to guide LLM output

**Few-Shot Solutions:**
- ✅ Consistent formatting through examples
- ✅ Control tone/style easily
- ✅ Include all expected sections
- ✅ No fine-tuning required
- ✅ Simple prompt engineering

### When to Use

✅ **Good for:**
- Need consistent answer format
- Specific style requirements
- Domain-specific conventions
- Template-based answers
- Customer-facing applications
- Brand voice consistency

❌ **Not ideal for:**
- Format doesn't matter
- Each query needs unique format
- No clear examples available
- Very simple queries
- When zero-shot works fine

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Example Selector]
    
    B --> C[Example Store<br/>Q&A Pairs]
    
    C --> D[Select Relevant<br/>Examples]
    
    D --> E[2-5 Example Pairs]
    
    A --> F[Vector Retrieval<br/>OpenSearch]
    
    F --> G[Retrieved Documents]
    
    E --> H[Build Prompt:<br/>Examples + Context + Query]
    G --> H
    
    H --> I[Generate Answer<br/>Following Pattern]
    
    I --> J[Formatted Answer]
    
    style A fill:#e1f5ff
    style C fill:#fff3e0
    style E fill:#e8f5e9
    style H fill:#f3e5f5
    style I fill:#c8e6c9
    style J fill:#b3e5fc
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Optional
import time
from dataclasses import dataclass

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

# Expected output:
# ✓ Imports successful

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'few_shot_rag_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
LLM_MODEL = 'us.anthropic.claude-sonnet-4-6'

# Few-Shot Parameters
NUM_EXAMPLES = 3  # Number of example Q&A pairs
RETRIEVAL_TOP_K = 3

print(f"Configuration:")
print(f"  Model: {LLM_MODEL.split('.')[-1][:40]}")
print(f"  Examples per query: {NUM_EXAMPLES}")
print(f"  Retrieval: Top-{RETRIEVAL_TOP_K}")

# Expected output:
# Configuration:
#   Model: claude-opus-4-1-20250805-v1:0
#   Examples per query: 3
#   Retrieval: Top-3

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

# Expected output:
# ✓ Services initialized

## 4️⃣ Example Q&A Pairs

Define examples that show desired answer format and style.

In [ ]:
@dataclass
class Example:
    """One Q&A example pair"""
    question: str
    answer: str
    category: str  # For selective retrieval

# Define example Q&A pairs showing desired format
example_library = [
    Example(
        question="What is AWS Bedrock?",
        answer="""**Overview:** AWS Bedrock is a fully managed service providing API access to foundation models.

**Key Features:**
- Serverless access to models (Claude, Titan, etc.)
- Pay-per-use pricing
- No infrastructure management

**Use When:** You need LLM capabilities without managing infrastructure.

**Cost:** Varies by model, Claude Opus: $15 input/$75 output per million tokens.""",
        category="service_explanation"
    ),
    Example(
        question="How does vector search work?",
        answer="""**Overview:** Vector search finds similar items by comparing high-dimensional embeddings.

**Key Features:**
- Converts text to vectors (embeddings)
- Uses similarity metrics (cosine, L2)
- Fast retrieval with HNSW algorithm

**Use When:** You need semantic search, not just keyword matching.

**Cost:** OpenSearch: ~$0.001 per query, embeddings: $0.0001 per 1K tokens.""",
        category="concept_explanation"
    ),
    Example(
        question="Should I use Opus or Haiku?",
        answer="""**Overview:** Choose based on your quality and cost requirements.

**Key Features:**
- Opus: Best quality, complex reasoning, $15/$75
- Haiku: Fast and cheap, simple tasks, $0.25/$1.25

**Use When:** 
- Opus: Complex analysis, critical accuracy
- Haiku: Simple queries, high volume

**Cost:** Haiku is 20x cheaper, use unless complexity demands Opus.""",
        category="comparison"
    ),
    Example(
        question="What are RAG patterns?",
        answer="""**Overview:** RAG patterns are different approaches to combining retrieval with generation.

**Key Features:**
- Simple RAG: Single retrieval, fast
- Self-RAG: Iterative refinement
- Ensemble: Multiple strategies

**Use When:** Different patterns suit different query types and requirements.

**Cost:** Ranges from $0.05 (Simple) to $0.20 (Ensemble) per query.""",
        category="concept_explanation"
    ),
    Example(
        question="How do I optimize RAG costs?",
        answer="""**Overview:** Multiple strategies can reduce RAG operational costs.

**Key Features:**
- Cache embeddings (reduce API calls 80%)
- Use Haiku for simple queries
- Limit retrieval to top-K needed

**Use When:** Running high-volume RAG systems in production.

**Cost:** Optimization can reduce per-query cost from $0.05 to $0.01.""",
        category="optimization"
    ),
]

print(f"✓ Loaded {len(example_library)} example Q&A pairs")
print("\nExample format:")
print(f"  - Overview section")
print(f"  - Key Features (bulleted)")
print(f"  - Use When guidance")
print(f"  - Cost information")

# Expected output:
# ✓ Loaded 5 example Q&A pairs
# 
# Example format:
#   - Overview section
#   - Key Features (bulleted)
#   - Use When guidance
#   - Cost information

## 5️⃣ Load Knowledge Base

In [ ]:
sample_documents = [
    "DynamoDB is a fully managed NoSQL database with single-digit millisecond performance at any scale.",
    "DynamoDB pricing: $0.25 per GB-month storage, $0.00013 per write, $0.000025 per read.",
    "DynamoDB supports both key-value and document data models with flexible schema.",
    "OpenSearch Serverless auto-scales compute and storage, eliminating cluster management.",
    "OpenSearch vector search uses HNSW algorithm for sub-100ms similarity search.",
    "OpenSearch pricing: Varies by workload, typically $0.24/hour for compute units.",
    "Claude Sonnet balances quality and cost at $3 input, $15 output per million tokens.",
    "Claude Haiku is optimized for speed and cost: $0.25 input, $1.25 output per million.",
    "Semantic chunking preserves context better than fixed-size chunking for RAG.",
    "Reranking adds LLM scoring after retrieval to improve precision.",
    "Fusion RAG generates multiple query variants and merges results with RRF.",
    "Few-shot learning uses examples to guide LLM output format and style.",
    "Cache frequently accessed embeddings in ElastiCache to reduce API costs.",
    "Use batch inference for 50% cost savings on async, non-latency-critical workloads.",
    "Monitor CloudWatch metrics: latency, cost per query, precision, recall.",
]

# Index documents
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

print("Indexing knowledge base...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })

opensearch.index_documents(documents)
print(f"✓ Indexed {len(documents)} documents")

# Expected output:
# Indexing knowledge base...
# ✓ Indexed 15 documents

## 6️⃣ Few-Shot RAG Pipeline

In [ ]:
def few_shot_rag(query: str,
                examples: List[Example],
                num_examples: int = 3,
                top_k: int = 3) -> Dict:
    """
    Few-shot RAG with example-guided generation.
    
    Returns:
        Dict with answer, examples_used, metadata
    """
    start_time = time.time()
    
    print(f"Query: {query}\n")
    print("="*70)
    
    # Step 1: Select examples
    print("\nStep 1: Example Selection")
    # For simplicity, use first N examples
    # In production, could select based on similarity to query
    selected_examples = examples[:num_examples]
    print(f"  Selected {len(selected_examples)} examples to guide generation")
    
    # Step 2: Retrieve documents
    print("\nStep 2: Document Retrieval")
    query_emb = embedder.embed_text(query)
    docs = opensearch.vector_search(query_emb, top_k=top_k)
    context = [d['text'] for d in docs]
    print(f"  Retrieved {len(context)} documents")
    
    # Step 3: Build few-shot prompt
    print("\nStep 3: Building Few-Shot Prompt")
    
    # Format examples
    examples_text = "Here are examples showing the desired answer format:\n\n"
    for i, ex in enumerate(selected_examples, 1):
        examples_text += f"Example {i}:\n"
        examples_text += f"Q: {ex.question}\n"
        examples_text += f"A: {ex.answer}\n\n"
    
    # Format context
    context_text = "Retrieved Information:\n"
    context_text += "\n".join([f"- {doc}" for doc in context])
    
    # Build full prompt
    few_shot_prompt = f"""
{examples_text}
{context_text}

Now answer the following question using the SAME FORMAT as the examples above:

Q: {query}
A:
"""
    
    print(f"  Prompt includes:")
    print(f"    - {len(selected_examples)} example Q&A pairs")
    print(f"    - {len(context)} retrieved documents")
    print(f"    - Format guidance")
    
    # Step 4: Generate answer
    print("\nStep 4: Generate Answer (following examples)")
    answer = llm.generate(few_shot_prompt, temperature=0.7)
    print(f"  Generated {len(answer)} chars")
    
    total_time = time.time() - start_time
    
    print(f"\n{'='*70}")
    print("COMPLETED")
    print(f"{'='*70}")
    print(f"  Total time: {total_time:.2f}s")
    print()
    
    return {
        'answer': answer,
        'examples_used': selected_examples,
        'retrieved_docs': context,
        'metadata': {
            'total_time': total_time,
            'num_examples': len(selected_examples),
            'num_docs': len(context)
        }
    }

print("✓ Few-shot RAG pipeline ready")

# Expected output:
# ✓ Few-shot RAG pipeline ready

## 7️⃣ Test Few-Shot RAG

In [ ]:
# Test query
test_query = "What is Amazon DynamoDB?"

print(f"\n{'#'*70}")
print(f"# FEW-SHOT RAG TEST")
print(f"{'#'*70}\n")

result = few_shot_rag(
    query=test_query,
    examples=example_library,
    num_examples=NUM_EXAMPLES,
    top_k=RETRIEVAL_TOP_K
)

print("="*70)
print("EXAMPLES PROVIDED")
print("="*70 + "\n")

for i, ex in enumerate(result['examples_used'], 1):
    print(f"Example {i}: {ex.question}")
    print(f"  Shows format with: Overview, Key Features, Use When, Cost\n")

print("="*70)
print("GENERATED ANSWER (Following Example Format)")
print("="*70)
print(f"\n{result['answer']}\n")

print("="*70)
print("FORMAT CONSISTENCY CHECK")
print("="*70)

# Check if answer follows format
has_overview = "**Overview:**" in result['answer']
has_features = "**Key Features:**" in result['answer']
has_use_when = "**Use When:**" in result['answer']
has_cost = "**Cost:**" in result['answer']

print(f"  ✓ Overview section: {has_overview}")
print(f"  ✓ Key Features section: {has_features}")
print(f"  ✓ Use When section: {has_use_when}")
print(f"  ✓ Cost section: {has_cost}")
print(f"\n  Format consistency: {sum([has_overview, has_features, has_use_when, has_cost])}/4 sections")

# Expected output:
# [Shows answer following the example format]

## 8️⃣ Comparison: Few-Shot vs Zero-Shot

In [ ]:
comparison_query = "Tell me about OpenSearch Serverless"

print("="*70)
print("FEW-SHOT RAG (With Examples)")
print("="*70 + "\n")

few_shot_result = few_shot_rag(
    query=comparison_query,
    examples=example_library,
    num_examples=NUM_EXAMPLES,
    top_k=RETRIEVAL_TOP_K
)

print("\n" + "="*70)
print("ZERO-SHOT RAG (No Examples)")
print("="*70 + "\n")

# Zero-shot: no examples
zero_shot_start = time.time()
query_emb = embedder.embed_text(comparison_query)
zero_docs = opensearch.vector_search(query_emb, top_k=RETRIEVAL_TOP_K)
zero_context = [d['text'] for d in zero_docs]
zero_answer = llm.generate_with_context(comparison_query, zero_context)
zero_shot_time = time.time() - zero_shot_start

print(f"Retrieved {len(zero_docs)} docs")
print(f"Generated answer in {zero_shot_time:.2f}s")

# Comparison
print("\n" + "="*70)
print("COMPARISON")
print("="*70)

print(f"\n📋 Formatting:")
print(f"  Few-Shot: Follows example structure (Overview, Features, Use When, Cost)")
print(f"  Zero-Shot: Free-form, no guaranteed structure")

print(f"\n📚 Guidance:")
print(f"  Few-Shot: {few_shot_result['metadata']['num_examples']} example Q&A pairs")
print(f"  Zero-Shot: No examples (general knowledge only)")

print(f"\n⏱️  Latency:")
print(f"  Few-Shot: {few_shot_result['metadata']['total_time']:.2f}s (includes examples in prompt)")
print(f"  Zero-Shot: {zero_shot_time:.2f}s")

print(f"\n💰 Cost (estimated):")
# Few-shot has longer prompt due to examples
few_shot_cost = 0.06  # ~20% more tokens due to examples
zero_shot_cost = 0.05
print(f"  Few-Shot: ~${few_shot_cost:.3f} (longer prompt with examples)")
print(f"  Zero-Shot: ~${zero_shot_cost:.3f} (baseline)")

print(f"\n✨ Consistency:")
print(f"  Few-Shot: High (follows learned pattern)")
print(f"  Zero-Shot: Variable (depends on model's interpretation)")

print(f"\n📝 Answers:\n")
print(f"Few-Shot (structured):\n{few_shot_result['answer'][:300]}...\n")
print(f"Zero-Shot (free-form):\n{zero_answer[:300]}...")

print(f"\n💡 Key Advantage:")
print(f"  Few-shot provides consistent formatting through examples,")
print(f"  ensuring all answers follow the same structure.")

# Expected output:
# [Shows few-shot's structured format vs zero-shot's free-form]

## 9️⃣ Summary & Key Takeaways

### What We Built

✅ Example-guided answer generation
✅ Consistent formatting across answers
✅ Example library with Q&A pairs
✅ Few-shot prompt construction
✅ Format consistency validation

### Performance Characteristics

- **Latency**: +10-20% vs zero-shot (longer prompt)
- **Cost**: +20% (example tokens in prompt)
- **Consistency**: Very high (follows pattern)
- **Quality**: Same or better (guided)
- **Setup**: Requires good examples

### Few-Shot vs Zero-Shot

| Aspect | Zero-Shot | Few-Shot |
|--------|-----------|----------|
| Examples | None | 2-5 Q&A pairs |
| Format | Variable | Consistent |
| Setup | None | Need examples |
| Cost | $0.05 | $0.06 |
| Latency | ~2s | ~2.3s |
| Consistency | Low | High |
| Best for | Flexible | Structured |

### When to Use Few-Shot

**Use Few-Shot when:**
- Need consistent format
- Specific style required
- Template-based answers
- Customer-facing apps
- Brand voice important
- Have good examples

**Skip Few-Shot when:**
- Format doesn't matter
- Each answer unique
- No clear examples
- Zero-shot works fine
- Minimize prompt length

### Key Insights

1. **Pattern Learning**: LLM infers from examples
2. **No Fine-Tuning**: Pure prompt engineering
3. **Consistency**: All answers follow pattern
4. **Minimal Cost**: Only prompt length increase
5. **Easy Setup**: Just need good examples

### Example Quality Matters

**Good Examples:**
- Clear structure
- Consistent format
- Representative
- Well-written
- Diverse but uniform

**Bad Examples:**
- Inconsistent formats
- Poorly written
- Too similar
- Confusing structure
- Contradictory patterns

### Best Practices

1. **2-5 Examples**: Sweet spot for learning
2. **Consistent Format**: All examples same structure
3. **Representative**: Cover common query types
4. **Clear Pattern**: Make format obvious
5. **Quality Examples**: Well-written and accurate
6. **Example Selection**: Can match to query (advanced)

### Example Selection Strategies

**Static (Simple):**
- Use same examples for all queries
- Fast and consistent
- Good when one format fits all

**Dynamic (Advanced):**
- Select examples similar to query
- Better relevance
- Requires example embeddings

**Category-Based:**
- Group examples by type
- Select from matching category
- Good middle ground

### Advanced Techniques

- **Semantic Example Selection**: Use embeddings to find similar examples
- **Multi-Format**: Different examples for different query types
- **Example Ranking**: Order examples by relevance
- **Dynamic Example Count**: More examples for complex queries
- **Example Caching**: Store formatted examples
- **A/B Testing**: Compare different example sets

### Limitations

1. **Longer Prompts**: +20% tokens from examples
2. **Example Dependency**: Quality depends on examples
3. **Limited Flexibility**: Constrains output format
4. **Setup Required**: Must curate examples
5. **Not Always Needed**: Zero-shot often sufficient

### Real-world Applications

**Where Few-Shot Excels:**
- Customer support (consistent tone)
- Product documentation (structured format)
- Technical Q&A (standard sections)
- Brand communications (voice consistency)
- Educational content (learning format)

### Cost Breakdown

**With 3 examples (~500 tokens):**
- Base query + context: ~1000 tokens input
- Examples: +500 tokens input
- Answer: ~400 tokens output
- **Total**: 1500 input + 400 output = ~$0.06
- **vs Zero-shot**: 1000 input + 400 output = ~$0.05

**Worth it?** Yes for customer-facing apps needing consistency

### Comparison with Fine-Tuning

**Few-Shot (Prompt Engineering):**
- Setup: Minutes (write examples)
- Cost: +20% per query
- Flexibility: Easy to change
- Best for: Format/style control

**Fine-Tuning:**
- Setup: Hours/days (training)
- Cost: Training + hosting
- Flexibility: Hard to change
- Best for: Domain knowledge

### Next Steps

- **Build Example Library**: Curate high-quality examples
- **Test Formats**: Try different structures
- **Dynamic Selection**: Implement similarity-based selection
- **Monitor Consistency**: Track format adherence
- **User Feedback**: Validate format preference

---

## 🎉 Phase 2 COMPLETE!

**Progress**: 21/37 patterns (57%)
- Phase 1: 10/10 ✅ Complete
- Phase 2: 11/12 ✅ Complete (LangGraph skipped)

**Phase 2 Complete Patterns:**
1. ✅ Multi-modal RAG
2. ✅ Agentic RAG
3. ✅ Corrective RAG (CRAG)
4. ✅ Self-RAG
5. ✅ Tree of Thoughts
6. ✅ Chain of Thought
7. ✅ ReAct
8. ✅ Memory-Augmented
9. ✅ Ensemble
10. ✅ Iterative
11. ✅ Few-Shot
12. ⏭️ LangGraph (skipped - AWS native only)

**Next**: Phase 3 - Specialized Patterns (10 patterns)

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")

# Expected output:
# 
# To delete the index later:
#   opensearch.delete_index('few_shot_rag_docs')